# **Filtrado de Recalls por Exclusión de Marcas C/E**

Este notebook filtra los datos de Recalls excluyendo las marcas que son exclusivas de Child Seats (C) y Equipment (E).

**Objetivos:**
1. Cargar lista de marcas exclusivas (727 marcas identificadas en Chunk 1)
2. Filtrar Recalls excluyendo esas marcas
3. Guardar archivo filtrado

In [6]:
import pandas as pd
from pathlib import Path
import json

# Configuración de rutas
BASE_DIR = Path(".." if Path.cwd().name == "notebooks" else ".")
IN_DIR = BASE_DIR / "data/processed"
OUT_DIR = BASE_DIR / "data/processed"

print(f"Working directory: {Path.cwd()}")
print(f"Base directory: {BASE_DIR}")

Working directory: c:\Users\moral\tec_final\notebooks
Base directory: ..


In [7]:
# PASO 1: Cargar marcas excluibles
print("="*70)
print("PASO 1: CARGA DE MARCAS EXCLUIBLES")
print("="*70)

with open(OUT_DIR / "excluded_makes_from_c_e.json", "r") as f:
    excluded_makes = set(json.load(f))

print(f"[OK] Marcas excluibles: {len(excluded_makes)}")
print(f"\nPrimeras 20 marcas excluibles:")
print(sorted(list(excluded_makes))[:20])

PASO 1: CARGA DE MARCAS EXCLUIBLES
[OK] Marcas excluibles: 727

Primeras 20 marcas excluibles:
['20TH CENTURY', '3M', '4WHEELS PARTS', 'A-1 ALTERNATIVE FUEL SYST', 'A.L. SOLUTIONS', 'A2ZEV', 'AAC', 'AAI MOTORSPORTS', 'ABADDON PRODUCTS', 'ABS-ACT-35 3-PT LWM 12', 'AC DELCO', 'ACC', 'ACCESSORY', 'ACCESSORY DISTRIBUTORS', 'ACCU-FAB', 'ACCU-FLOW', 'ACCURIDE', 'ACD TRIDON', 'ACE', 'ACE ELECTRIC']


In [8]:
# PASO 2: Cargar y filtrar RECALLS
print("\n" + "="*70)
print("PASO 2: FILTRADO DE RECALLS")
print("="*70)

df_recalls = pd.read_csv(IN_DIR / "recalls_by_campaign.csv")
print(f"Recalls originales: {len(df_recalls):,}")
print(f"Columnas: {df_recalls.columns.tolist()}")

# Normalizar marca
df_recalls['MAKE_NORMALIZED'] = df_recalls['MAKETXT'].astype(str).str.strip().str.upper()

# Verificación: cuántas marcas C/E hay originalmente
makes_in_data = set(df_recalls['MAKE_NORMALIZED'].unique())
matches = makes_in_data & excluded_makes
print(f"\nMarcas C/E en datos originales: {len(matches)}")
print(f"Marcas C/E encontradas: {sorted(list(matches))[:20] if matches else 'Ninguna'}")

# Filtrar
df_recalls_filtered = df_recalls[~df_recalls['MAKE_NORMALIZED'].isin(excluded_makes)].copy()
df_recalls_filtered = df_recalls_filtered.drop(columns=['MAKE_NORMALIZED'])

print(f"\nRecalls filtrados: {len(df_recalls_filtered):,}")
print(f"Recalls excluidos: {len(df_recalls) - len(df_recalls_filtered):,}")
print(f"Reducción: {100 * (len(df_recalls) - len(df_recalls_filtered)) / len(df_recalls):.1f}%")

# Verificación post-filtro
makes_filtered = set(df_recalls_filtered['MAKETXT'].astype(str).str.strip().str.upper().unique())
matches_filtered = makes_filtered & excluded_makes
print(f"\n[VERIFICACION] Marcas C/E en datos filtrados: {len(matches_filtered)}")
print(f"[VERIFICACION] {'EXITO: No quedan marcas C/E' if len(matches_filtered) == 0 else 'ERROR: Aun hay marcas C/E'}")


PASO 2: FILTRADO DE RECALLS
Recalls originales: 13,653
Columnas: ['CAMPNO', 'POTAFF_num', 'MAKETXT', 'MODELTXT', 'COMPNAME', 'COMP_L1', 'YEARTXT']

Marcas C/E en datos originales: 0
Marcas C/E encontradas: Ninguna

Recalls filtrados: 13,653
Recalls excluidos: 0
Reducción: 0.0%

[VERIFICACION] Marcas C/E en datos filtrados: 0
[VERIFICACION] EXITO: No quedan marcas C/E


In [4]:
# PASO 3: Guardar
df_recalls_filtered.to_csv(OUT_DIR / "recalls_filtered.csv", index=False)

print("\n" + "="*70)
print("ARCHIVO GUARDADO")
print("="*70 + "\n")
print(f"Recalls filtrados: {len(df_recalls_filtered):,} registros")
print(f"Archivo: {OUT_DIR / 'recalls_filtered.csv'}")
print("\n" + "="*70)
print("FILTRADO DE RECALLS COMPLETADO")
print("="*70)


ARCHIVO GUARDADO

Recalls filtrados: 13,653 registros
Archivo: ..\data\processed\recalls_filtered.csv

FILTRADO DE RECALLS COMPLETADO
